# 02 — Feature Engineering

This notebook handles data cleaning and feature engineering for the Apple Watch bridge layer.

**Calls:** `src/preprocess.py`  
**Input:** `data/apple_watch/*_raw.csv`  
**Output:** `data/processed/*_clean.csv`

In [ ]:
import sys
import os
import pandas as pd

sys.path.append(r'C:\Projects\GA Capstone Project')
from src.preprocess import run_cleaning_pipeline

RAW_DIR  = r'C:\Projects\GA Capstone Project\data\apple_watch'
PROC_DIR = r'C:\Projects\GA Capstone Project\data\processed'

print("="*65)
print("APPLE WATCH DATA CLEANING PIPELINE")
print("02_feature_engineering.ipynb")
print("="*65)

reports = run_cleaning_pipeline(RAW_DIR, PROC_DIR)

print(f"\n{'METRIC':<22} {'INITIAL':>10} "
      f"{'REMOVED':>10} {'FINAL':>10} {'RETAINED':>10}")
print("-"*65)

for r in reports:
    removed = r['non_numeric'] + r['outliers_removed']
    print(f"{r['metric']:<22} "
          f"{r['initial_records']:>10,} "
          f"{removed:>10,} "
          f"{r['final_records']:>10,} "
          f"{r['retention_pct']:>9.1f}%")

print("="*65)
print("\nCleaned files saved to data/processed/")
print("Anchor date: June 18 2025")
print("\nAnchor period definitions:")
print("  baseline    \u2014 more than 91 days before anchor")
print("  pre_anchor  \u2014 within 90 days before anchor")
print("  post_anchor \u2014 within 90 days after anchor")
print("  follow_up   \u2014 more than 90 days after anchor")

## Quick Check — Heart Rate Distribution & Anchor Periods

Inspection only — no `src/` calls. Validates that cleaned heart rate data has a realistic distribution and confirms anchor period sample sizes for Layer 2 analysis.

In [ ]:
import pandas as pd
import os

PROC_DIR = r'C:\Projects\GA Capstone Project\data\processed'

hr = pd.read_csv(os.path.join(PROC_DIR, 'heart_rate_clean.csv'))

print("Heart Rate Distribution Check")
print("="*40)
print(f"Min:    {hr['value'].min():.1f} bpm")
print(f"Max:    {hr['value'].max():.1f} bpm")
print(f"Mean:   {hr['value'].mean():.1f} bpm")
print(f"Median: {hr['value'].median():.1f} bpm")
print(f"Std:    {hr['value'].std():.1f} bpm")
print(f"\nReadings below 40 bpm: "
      f"{(hr['value'] < 40).sum():,}")
print(f"Readings above 200 bpm: "
      f"{(hr['value'] > 200).sum():,}")
print(f"\nAnchor period distribution:")
print(hr['anchor_period'].value_counts().to_string())

In [ ]:
edge_cases = hr[
    (hr['value'] < 40) | (hr['value'] > 200)
][['startDate', 'value', 'hour', 'anchor_period']]

print("Edge Case Readings")
print("="*55)
print(edge_cases.to_string())